# PatchTST 机械臂扭矩预测训练

## 说明
本notebook使用PatchTST模型（ICLR 2023）进行机械臂扭矩曲线预测。

**核心技术**：
- **PatchTST**: 最先进的时间序列预测架构
- **Residual Learning**: 预测差分值而非绝对值
- **Channel Independence**: 每个信号独立建模

**数据格式**：
- 文件名: `processed_Data_a_b_open.csv`
- 列名: `Time(s), T, Fx, Fy, Fz`
- 预测目标: Fy (索引1)

**预期效果**：
- R² > 0.5 (vs LSTM的-0.05)
- MAPE < 30% (vs LSTM的659%)

## 1. 环境设置

In [ ]:
# 导入必要的库
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
from IPython.display import display, HTML

# 设置中文字体和绘图风格
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 负号显示
plt.rcParams['figure.figsize'] = (12, 6)
plt.style.use('seaborn-v0_8-darkgrid')

# 检查CUDA
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. 配置参数

In [ ]:
# ==================== 数据参数 ====================
DATA_DIR = './data'
PATTERN = 'processed_*.csv'  # 新的文件名格式
TRAIN_SPLIT = 0.8

# ==================== 序列长度（PatchTST需要固定长度）====================
SEQ_LEN = 2000      # 输入序列长度
PRED_LEN = 1000     # 预测长度
STEP_SIZE = 500     # 滑动窗口步长（数据增强）

# ==================== PatchTST模型参数 ====================
INPUT_DIM = 3       # 3个力传感器信号 (Fx, Fy, Fz)
D_MODEL = 128       # Transformer维度
N_HEADS = 8         # 注意力头数
E_LAYERS = 3        # Encoder层数
PATCH_LEN = 32      # Patch长度
STRIDE = 16         # Patch步长
DROPOUT = 0.2

# ==================== 训练参数 ====================
BATCH_SIZE = 32
LEARNING_RATE = 0.0005  # Transformer通常使用较小学习率
EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15

# ==================== 保存目录 ====================
SAVE_DIR = './models_patchtst'
RESULTS_DIR = './results_patchtst'

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("\n配置参数：")
print(f"  数据目录: {DATA_DIR}")
print(f"  序列长度: {SEQ_LEN} → {PRED_LEN}")
print(f"  Patch配置: patch_len={PATCH_LEN}, stride={STRIDE}")
print(f"  模型配置: d_model={D_MODEL}, heads={N_HEADS}, layers={E_LAYERS}")
print(f"  训练配置: batch_size={BATCH_SIZE}, lr={LEARNING_RATE}, epochs={EPOCHS}")

## 3. 加载数据

使用专门为PatchTST设计的固定长度数据加载器。

**特点**：
- 固定输入/输出长度
- 保留Residual Learning（返回差分值）
- 按瓶子编号划分train/test（防止数据泄漏）

In [ ]:
from src.data_loader import load_data_for_patchtst

train_loader, test_loader = load_data_for_patchtst(
    data_dir=DATA_DIR,
    pattern=PATTERN,
    train_split=TRAIN_SPLIT,
    seq_len=SEQ_LEN,
    pred_len=PRED_LEN,
    signal_type='Fy',  # 预测Fy信号
    use_all_features=True,  # 使用Fx, Fy, Fz三个特征
    batch_size=BATCH_SIZE,
    step_size=STEP_SIZE
)

print(f"\n✅ 数据加载完成！")
print(f"训练批次: {len(train_loader)}")
print(f"测试批次: {len(test_loader)}")

## 4. 创建PatchTST模型

PatchTST架构：
1. **Patch Embedding**: 将2000步切分成~123个patches
2. **Transformer Encoder**: 3层multi-head attention
3. **Channel Independence**: 每个信号独立处理
4. **Prediction Head**: 输出1000步预测

In [ ]:
from src.model import get_model

model = get_model(
    model_type='patchtst',
    input_dim=INPUT_DIM,
    hidden_dim=D_MODEL,
    num_layers=E_LAYERS,
    output_length=PRED_LEN,
    dropout=DROPOUT,
    seq_len=SEQ_LEN,
    patch_len=PATCH_LEN,
    stride=STRIDE,
    n_heads=N_HEADS
)

# 计算patch数量
num_patches = (SEQ_LEN - PATCH_LEN) // STRIDE + 1
print(f"\nPatch信息：")
print(f"  原始序列长度: {SEQ_LEN}")
print(f"  切分后patches数: {num_patches}")
print(f"  复杂度降低: {SEQ_LEN / num_patches:.1f}x")

## 5. 训练模型

训练过程：
- 优化器: Adam
- 学习率调度: ReduceLROnPlateau
- 早停: 15个epoch无改进则停止
- 保存: 保存最佳模型到 `models_patchtst/best_model.pth`

In [ ]:
from src.train import Trainer

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    device=device,
    learning_rate=LEARNING_RATE,
    save_dir=SAVE_DIR,
    teacher_forcing_ratio=0.0  # PatchTST不使用Teacher Forcing
)

print("\n开始训练...")
print("=" * 80)

history = trainer.train(
    epochs=EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE
)

print("\n✅ 训练完成！")

## 6. 训练曲线可视化

In [ ]:
# 绘制训练曲线
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss曲线
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['test_loss'], label='Test Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('训练/测试损失曲线', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# 学习率曲线
axes[1].plot(history['learning_rate'], color='red', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Learning Rate', fontsize=12)
axes[1].set_title('学习率变化', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\n最佳测试损失: {min(history['test_loss']):.6f}")
print(f"最终学习率: {history['learning_rate'][-1]:.6f}")

## 7. 评估Baseline模型

**Baseline**: Persistence Model（持久化预测）
- 策略: 用输入序列的最后一个Fy值预测所有输出
- 目的: 提供最简单的对比基准

In [ ]:
from src.evaluate import evaluate_baseline

baseline_metrics = evaluate_baseline(test_loader, device=device)

print("\nBaseline模型评估结果：")
print("=" * 60)
for key, value in baseline_metrics.items():
    print(f"  {key:10s}: {value:.6f}")
print("=" * 60)

## 8. 评估PatchTST模型

加载最佳模型并评估性能。

**注意**: 评估时会自动将预测的差分值重建为绝对值。

In [ ]:
from src.evaluate import evaluate_model

# 加载最佳模型
trainer.load_checkpoint('best_model.pth')

# 评估
patchtst_metrics, predictions, targets, inputs = evaluate_model(
    model=trainer.model,
    data_loader=test_loader,
    device=device,
    save_dir=RESULTS_DIR
)

print("\nPatchTST模型评估结果：")
print("=" * 60)
for key, value in patchtst_metrics.items():
    print(f"  {key:10s}: {value:.6f}")
print("=" * 60)

## 9. 结果对比分析

对比Baseline和PatchTST的性能。

In [ ]:
# 创建对比表格
import pandas as pd

comparison_data = {
    '指标': ['R²', 'RMSE', 'MAE', 'MAPE'],
    'Baseline': [
        f"{baseline_metrics['R2']:.4f}",
        f"{baseline_metrics['RMSE']:.4f}",
        f"{baseline_metrics['MAE']:.4f}",
        f"{baseline_metrics['MAPE']:.2f}%"
    ],
    'PatchTST': [
        f"{patchtst_metrics['R2']:.4f}",
        f"{patchtst_metrics['RMSE']:.4f}",
        f"{patchtst_metrics['MAE']:.4f}",
        f"{patchtst_metrics['MAPE']:.2f}%"
    ]
}

# 计算改进
r2_improve = ((patchtst_metrics['R2'] - baseline_metrics['R2']) / abs(baseline_metrics['R2']) * 100) if baseline_metrics['R2'] != 0 else float('inf')
rmse_improve = ((baseline_metrics['RMSE'] - patchtst_metrics['RMSE']) / baseline_metrics['RMSE'] * 100)
mae_improve = ((baseline_metrics['MAE'] - patchtst_metrics['MAE']) / baseline_metrics['MAE'] * 100)

comparison_data['改进'] = [
    f"{r2_improve:+.1f}%",
    f"{rmse_improve:+.1f}%",
    f"{mae_improve:+.1f}%",
    f"—"
]

df_comparison = pd.DataFrame(comparison_data)

print("\n" + "=" * 80)
print("最终对比结果")
print("=" * 80)
print(df_comparison.to_string(index=False))
print("=" * 80)

# 可视化对比
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

metrics_names = ['R²', 'RMSE', 'MAE']
baseline_vals = [baseline_metrics['R2'], baseline_metrics['RMSE'], baseline_metrics['MAE']]
patchtst_vals = [patchtst_metrics['R2'], patchtst_metrics['RMSE'], patchtst_metrics['MAE']]

x = np.arange(len(metrics_names))
width = 0.35

bars1 = ax.bar(x - width/2, baseline_vals, width, label='Baseline', alpha=0.8)
bars2 = ax.bar(x + width/2, patchtst_vals, width, label='PatchTST', alpha=0.8)

ax.set_ylabel('数值', fontsize=12)
ax.set_title('Baseline vs PatchTST 性能对比', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

## 10. 结论

总结PatchTST的表现：

In [ ]:
print("\n" + "=" * 80)
print("训练总结")
print("=" * 80)

r2_value = patchtst_metrics['R2']

if r2_value > 0.7:
    conclusion = "🎉 优秀！PatchTST达到了非常好的预测效果。"
elif r2_value > 0.5:
    conclusion = "✅ 良好！PatchTST显著优于Baseline，预测质量达到实用级别。"
elif r2_value > 0.3:
    conclusion = "📈 有效！PatchTST比Baseline好，但还有提升空间。建议调整超参数。"
elif r2_value > 0:
    conclusion = "⚠️ 及格。模型有效但表现一般，建议尝试增大模型容量或缩短预测长度。"
else:
    conclusion = "❌ 需要改进。建议检查数据质量，调整模型参数，或尝试其他方法。"

print(f"\nPatchTST R²: {r2_value:.4f}")
print(f"Baseline R²: {baseline_metrics['R2']:.4f}")
print(f"\n评价: {conclusion}")

print(f"\n模型保存位置: {SAVE_DIR}/best_model.pth")
print(f"结果保存位置: {RESULTS_DIR}/")
print("=" * 80)

## 11. (可选) 超参数调优建议

如果结果不理想，可以尝试以下调整：

In [ ]:
print("\n超参数调优建议：\n")

if r2_value < 0.3:
    print("1. 增大模型容量：")
    print("   - D_MODEL = 256 (从128增大)")
    print("   - N_HEADS = 16 (从8增大)")
    print("   - E_LAYERS = 4 (从3增大)")
    print("")
    print("2. 缩短预测长度：")
    print("   - PRED_LEN = 500 (从1000减小)")
    print("")
    print("3. 增加训练轮数：")
    print("   - EPOCHS = 200 (从100增大)")
    print("")
    print("4. 调整学习率：")
    print("   - LEARNING_RATE = 0.0001 (更小的学习率)")

elif r2_value < 0.5:
    print("1. 适当增大模型：")
    print("   - D_MODEL = 192")
    print("   - N_HEADS = 12")
    print("")
    print("2. 增加数据增强：")
    print("   - STEP_SIZE = 200 (从500减小，创建更多样本)")
    print("")
    print("3. 延长训练：")
    print("   - EPOCHS = 150")
    print("   - EARLY_STOPPING_PATIENCE = 20")

else:
    print("当前结果已经很好！如果想进一步提升：")
    print("")
    print("1. 模型集成：")
    print("   - 训练多个PatchTST模型，使用不同的随机种子")
    print("   - 对预测结果取平均")
    print("")
    print("2. 数据预处理优化：")
    print("   - 移除异常值")
    print("   - 使用更复杂的特征工程")
    print("")
    print("3. 超参数微调：")
    print("   - 使用网格搜索或贝叶斯优化")
    print("   - 尝试不同的patch_len和stride组合")